# Day 11: NumPy Vectorization and Broadcasting

**Dataset:** `Iris_Dataset.csv`

This notebook performs arithmetic operations on complete NumPy arrays and explores
broadcasting between arrays of compatible shapes, using real values pulled directly from the
Iris dataset.

## 1. Import Libraries and Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import time

# Load the dataset
df = pd.read_csv("../datasets/Iris_Dataset.csv")

# Preview the data
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


**Create a 1D array and a 2D array from real columns**

In [ ]:
petal_length = np.array(df["petal length (cm)"])

feature_columns = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
features = np.array(df[feature_columns])

print("1D array shape:", petal_length.shape)
print("2D array shape:", features.shape)

1D array shape: (150,)
2D array shape: (150, 4)


## 2. Examples of Vectorized Arithmetic

A vectorized operation applies to every element of an array at once, with no explicit loop.

**Example 1: Add a scalar to every value**

In [ ]:
petal_length_plus_1 = petal_length + 1
print(petal_length_plus_1[:10])

[2.4 2.4 2.3 2.5 2.4 2.7 2.4 2.5 2.4 2.5]


**Example 2: Multiply every value by a scalar**

In [ ]:
petal_length_cm_to_mm = petal_length * 10
print(petal_length_cm_to_mm[:10])

[14. 14. 13. 15. 14. 17. 14. 15. 14. 15.]


**Example 3: Element-wise arithmetic between two arrays of the same shape**

In [ ]:
sepal_length = np.array(df["sepal length (cm)"])
sepal_width = np.array(df["sepal width (cm)"])

sepal_area_approx = sepal_length * sepal_width
print(sepal_area_approx[:10])

[17.85 14.7  15.04 14.26 18.   21.06 15.64 17.   12.76 15.19]


**Example 4: Applying a NumPy math function to an entire array**

In [ ]:
petal_length_squared = np.square(petal_length)
petal_length_sqrt = np.sqrt(petal_length)

print(petal_length_squared[:5])
print(petal_length_sqrt[:5])

[1.96 1.96 1.69 2.25 1.96]
[1.18321596 1.18321596 1.14017543 1.22474487 1.18321596]


## 3. Broadcasting Examples

Broadcasting lets NumPy perform operations between arrays of **different but compatible
shapes**, without writing a loop or manually repeating one of the arrays.

**Example 1: A scalar (shape `()`) broadcast across a 1D array (shape `(150,)`)**

The single scalar value is applied to every one of the 150 elements.

In [ ]:
result = petal_length * 2
print("Array shape:", petal_length.shape, " Scalar shape: () ", "-> Result shape:", result.shape)
print(result[:5])

Array shape: (150,)  Scalar shape: ()  -> Result shape: (150,)
[2.8 2.8 2.6 3.  2.8]


**Example 2: A 1D array (shape `(4,)`) broadcast across a 2D array (shape `(150, 4)`)**

A single row of 4 values (the mean of each column) is broadcast across all 150 rows, so it's
subtracted from every row at once.

In [ ]:
column_means = features.mean(axis=0)
print("features shape:", features.shape, " column_means shape:", column_means.shape)

centered_features = features - column_means
print(centered_features[:5])

features shape: (150, 4)  column_means shape: (4,)
[[-0.74333333  0.446      -2.35866667 -0.99866667]
 [-0.94333333 -0.054      -2.35866667 -0.99866667]
 [-1.14333333  0.146      -2.45866667 -0.99866667]
 [-1.24333333  0.046      -2.25866667 -0.99866667]
 [-0.84333333  0.546      -2.35866667 -0.99866667]]


**Example 3: A column vector (shape `(150, 1)`) broadcast across a 2D array (shape `(150, 4)`)**

Reshaping a 1D array into a column lets it be broadcast across every *column*, instead of
every row, applying a different value to each row.

In [ ]:
petal_length_column = petal_length.reshape(-1, 1)
print("features shape:", features.shape, " petal_length_column shape:", petal_length_column.shape)

ratio_to_petal_length = features / petal_length_column
print(ratio_to_petal_length[:5])

features shape: (150, 4)  petal_length_column shape: (150, 1)
[[3.64285714 2.5        1.         0.14285714]
 [3.5        2.14285714 1.         0.14285714]
 [3.61538462 2.46153846 1.         0.15384615]
 [3.06666667 2.06666667 1.         0.13333333]
 [3.57142857 2.57142857 1.         0.14285714]]


## 4. Comparing Loop-Based and Vectorized Calculations

Both approaches below calculate the same thing (squaring every petal length value and summing
the result), but written two very different ways.

In [ ]:
# Loop-based version
start_time = time.time()

loop_result = 0
for value in petal_length:
    loop_result += value ** 2

loop_time = time.time() - start_time

# Vectorized version
start_time = time.time()

vectorized_result = np.sum(petal_length ** 2)

vectorized_time = time.time() - start_time

print("Loop result:", loop_result)
print("Vectorized result:", vectorized_result)
print("Results match:", np.isclose(loop_result, vectorized_result))
print(f"Loop time: {loop_time:.6f} seconds")
print(f"Vectorized time: {vectorized_time:.6f} seconds")

Loop result: 2583.0000000000005
Vectorized result: 2583.0
Results match: True
Loop time: 0.000110 seconds
Vectorized time: 0.000061 seconds


**Same comparison on a larger array, to make the speed difference clearer**

150 values is too small to show a meaningful timing gap, so this repeats the same calculation
on a much larger array built from the real data.

In [ ]:
large_array = np.tile(petal_length, 10000)   # repeat the real petal length data 10,000 times
print("Large array size:", large_array.shape[0])

# Loop-based version
start_time = time.time()

loop_result = 0
for value in large_array:
    loop_result += value ** 2

loop_time = time.time() - start_time

# Vectorized version
start_time = time.time()

vectorized_result = np.sum(large_array ** 2)

vectorized_time = time.time() - start_time

print("Results match:", np.isclose(loop_result, vectorized_result))
print(f"Loop time: {loop_time:.6f} seconds")
print(f"Vectorized time: {vectorized_time:.6f} seconds")
print(f"Vectorized version was about {loop_time / vectorized_time:.1f}x faster")

Large array size: 1500000


Results match: True
Loop time: 0.241454 seconds
Vectorized time: 0.006686 seconds
Vectorized version was about 36.1x faster


## Outcome

This notebook practiced vectorized arithmetic and broadcasting using real values from the
`Iris_Dataset.csv` file. I performed vectorized operations including adding and multiplying by
a scalar, element-wise arithmetic between two arrays, and applying NumPy math functions
(`np.square()`, `np.sqrt()`) across an entire array at once. I then demonstrated three
broadcasting examples: a scalar broadcast across a 1D array, a 1D array of column means
broadcast across every row of a 2D array, and a reshaped column vector broadcast across every
column of a 2D array. Finally, comparing a loop-based calculation against its vectorized
equivalent (i.e., on both the real 150-row dataset and a much larger repeated version of it) showed
that both approaches produce identical results, but the vectorized version runs substantially
faster, which is why avoiding explicit Python loops matters for real numerical work.